# Imports

In [1]:
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import os
import ollama

In [5]:
!pip install pypdf

In [2]:
!pip install ollama

In [3]:
!pip install faiss-cpu

   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
    --------------------------------------- 0.3/18.9 MB ? eta -:--:--
    --------------------------------------- 0.3/18.9 MB ? eta -:--:--
    --------------------------------------- 0.3/18.9 MB ? eta -:--:--
    --------------------------------------- 0.3/18.9 MB ? eta -:--:--
    --------------------------------------- 0.3/18.9 MB ? eta -:--:--
   - -------------------------------------- 0.5/18.9 MB 258.0 kB/s eta 0:01:12
   - -------------------------------------- 0.5/18.9 MB 258.0 kB/s eta 0:01:12
   - -------------------------------------- 0.5/18.9 MB 258.0 kB/s eta 0:01:12
   - -------------------------------------- 0.5/18.9 MB 258.0 kB/s eta 0:01:12
   - -------------------------------------- 0.5/18.9 M

In [7]:
!pip install sentence_transformers

  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
   ---------------------------------------- 0.0/10.1 MB ? eta -:--:--
   -- ------------------------------------- 0.5/10.1 MB 2.4 MB/s eta 0:00:05
   ------- -------------------------------- 1.8/10.1 MB 4.6 MB/s eta 0:00:02
   ------------- -------------------------- 3.4/10.1 MB 5.9 MB/s eta 0:00:02
   ------------- -------------------------- 3.4/10.1 MB 5.9 MB/s eta 0:00:02
   ----------------------- ---------------- 6.0/10.1 MB 5.7 MB/s eta 0:00:01
   ------------------------- -------------- 6.6/10.1 MB 5.8 MB/s eta 0:00:01
   -------------------------- ------------- 6.8/10.1 MB 5.0 MB/s eta 0:00:01
   ---------------------------------------  10.0/10.1 MB 6.3 MB/s eta 0:00:01
   ---------------------------------------- 10.1/10.1 MB 5.7 MB/s  0:00:01
   ---------------------------------------- 0.0/536.7 kB ? eta -:--:--
   ---------------------------------------- 0.0/536.7 kB ? eta -:--:--
   ------------------

# Load data

In [2]:
def extract_text_from_pdf(path):
    reader = PdfReader(path)
    pages = []
    for page in reader.pages:
        text = page.extract_text()
        if text:
            pages.append(text)
    return "\n".join(pages)

In [3]:
pdf_dir = "Статьи"
articles = []

for file in os.listdir(pdf_dir):
    if file.endswith(".pdf"):
        full_path = os.path.join(pdf_dir, file)
        text = extract_text_from_pdf(full_path)
        articles.append({"source": file, "text": text})

print("Статей загружено:", len(articles))

Статей загружено: 10


In [4]:
def chunk_text(text, chunk_size=500, overlap=100):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = words[start:end]
        chunks.append(" ".join(chunk))
        start += chunk_size - overlap
    return chunks

In [34]:
documents = []

for art in articles:
    chunks = chunk_text(art["text"], chunk_size=500, overlap=100)
    for ch in chunks:
        documents.append({
            "source": art["source"],
            "text": ch,
            "chunk_id": len(documents)
        })

print("Всего чанков:", len(documents))

Всего чанков: 243


In [131]:
model_emb = SentenceTransformer("all-MiniLM-L6-v2")
texts = [d["text"] for d in documents]
# embeddings = model_emb.encode(texts, show_progress_bar=True)
# embeddings = np.array(embeddings).astype("float32")

embeddings = model_emb.encode(texts, show_progress_bar=True)
embeddings = np.array(embeddings).astype("float32")
faiss.normalize_L2(embeddings)


print("Размер embedding:", embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Размер embedding: (243, 384)


Другие индексы:

IndexIVFFlat

IndexIVFPQ

HNSW

IndexIVFFlat — «кластеризация + точный поиск внутри»

Идея

IVF = Inverted File

Пространство разбивается на nlist кластеров (k-means)

Каждый вектор принадлежит одному кластеру

При поиске:

находим nprobe ближайших кластеров

ищем точно (L2 / IP) только внутри них



IndexIVFPQ — «кластеризация + сжатие векторов»

Идея

IVF + Product Quantization

Внутри каждого кластера:

векторы не хранятся полностью

они кодируются короткими кодами (например 8–16 байт)

Что такое PQ (очень кратко)

вектор разбивается на m частей

каждая часть квантуется отдельно

расстояния считаются по таблицам

In [132]:
dim = embeddings.shape[1]
# index = faiss.IndexFlatL2(dim)
index = faiss.IndexFlatIP(dim)
index.add(embeddings)

print("Векторов в индексе:", index.ntotal)

Векторов в индексе: 243


# RAG

In [37]:
def analyze_question(question: str):
    q = question.lower()
    return {
        "fact": any(w in q for w in ["which", "who", "where", "institution", "acknowledg"]),
        "numeric": any(w in q for w in ["percentage", "variance", "sd", "mean", "stability"]),
        "results": any(w in q for w in ["result", "experiment", "stability", "variance"]),
    }


In [133]:
def retrieve(query, top_k=10, source_filter=None):
    """
    Поиск релевантных чанков по embedding.
    query: текст вопроса
    top_k: количество возвращаемых чанков
    source: опционально ограничить одной статьей (по имени файла)
    """
    q_emb = model_emb.encode([query]).astype("float32")
    faiss.normalize_L2(q_emb)

    distances, indices = index.search(q_emb, top_k)

    results = []
    for i in indices[0]:
        doc = documents[i]
        if source_filter is None or doc["source"] == source_filter:
            results.append(doc)

    return results

In [52]:
def retrieve_auto(question, source=None):
    """
    Автоматический RAG retrieval без фильтрации по секциям.
    question: текст вопроса
    source: ограничить одной статьей (опционально)
    """
    flags = analyze_question(question)
    top_k = 20 if (flags.get("fact") or flags.get("numeric")) else 10
    docs = retrieve(question, top_k=top_k, source_filter=source)


    return docs

In [95]:
BASE_PROMPT_GENERAL = """
Answer the question using ONLY the provided context from the scientific article.
Do NOT use any external knowledge or general background theory.
Report numerical values and specific terms exactly as stated.
If the answer is not present in the context, say so explicitly.
Be concise but complete, following the structure of the article.

Context:
{context}

Question:
{question}

Answer:
"""

In [96]:
BASE_PROMPT_MATH = """
You must answer by RESTATING the argument given in the text.

Rules:
- Use only objects, definitions, identities, and logical steps that explicitly appear in the provided context.
- Do NOT introduce intuition, interpretations, analogies, or external mathematical facts.
- Do NOT replace additive sum collisions with equalities of single terms.
- Preserve the structure of the proof: definitions → identities → conclusion.
- If an equality or construction is central to the proof, state it explicitly.
- If the text does not justify a claim, do not include it.

Context:
{context}

Question:
{question}

Answer:
"""


In [161]:
BASE_PROMPT_MATH_STRICT = """
You must answer by STRICTLY RESTATING what is written in the provided context.

Rules:
- Use ONLY terminology, notation, definitions, and constructions that explicitly appear in the context.
- Do NOT explain concepts, define standard terms, or add background knowledge.
- Do NOT interpret notation (e.g. do NOT explain what “π₀”, “exact”, or “Symp” mean).
- Do NOT paraphrase definitions: restate them in minimal syntactic form, preserving all symbols and equalities.
- If a group, object, or map is defined by an equality, quotient, or formula, you MUST state that expression explicitly.
- Do NOT infer unstated properties or meanings.
- If a term is not explicitly defined in the context, do NOT define or explain it.
- You are not allowed to introduce standard mathematical facts.
- If the answer is not explicitly stated in the context, reply EXACTLY:
  "The context does not state this."

Output structure (and nothing else):
1. Definitions exactly as stated.
2. Constructions or maps exactly as stated.

Context:
{context}

Question:
{question}

Answer:
"""


In [117]:
def rag_answer(question, source=None, mode="general"):
    """
    Генерация ответа с использованием RAG.
    question: вопрос
    model_name: модель Ollama
    source: ограничение одной статьей
    mode: в зависимости от темы статьи
    """
    docs = retrieve_auto(question, source=source)
    
    if not docs:
        return "No relevant context found in the dataset."
        
    context = "\n\n".join([f"[{d['source']} | chunk {d['chunk_id']}]\n{d['text']}" for d in docs])

    if mode == "math":
        prompt = BASE_PROMPT_MATH.format(context=context, question=question)
        
    elif mode == "math_strict":
        prompt = BASE_PROMPT_MATH_STRICT.format(context=context, question=question)
        
    else:
        prompt = BASE_PROMPT_GENERAL.format(context=context, question=question)

    response = ollama.chat(
        model="qwen3:8b",
        messages=[{"role": "user", "content": prompt}]
    )

    return response["message"]["content"]


In [55]:
def answer_without_rag(question, model_name="qwen3:8b"):
    response = ollama.chat(
        model=model_name,
        messages=[
            {"role": "system", "content": "Answer concisely and factually."},
            {"role": "user", "content": question}
        ]
    )
    return response["message"]["content"]

# 1 Distinguishing the nature of dark matter by mapping cosmic filaments from Lyman-alpha emission

In [20]:
# 1 https://arxiv.org/pdf/2601.22677

=== Без RAG ===
The main idea behind using Lyman-alpha emission to distinguish the nature of dark matter in the study is to leverage the relationship between dark matter's gravitational influence and the distribution of hydrogen gas, which emits Lyman-alpha radiation. Here's a structured breakdown:

1. **Cosmic Filaments as Tracers**: Cosmic filaments are vast, thread-like structures in the universe composed of dark matter and baryonic gas (hydrogen). These structures are shaped by dark matter's gravitational pull. By mapping Lyman-alpha emission—light emitted by hydrogen atoms transitioning between energy levels—researchers can trace the distribution of hydrogen gas along these filaments.

2. **Dark Matter's Structural Impact**: Different dark matter models (e.g., Cold Dark Matter [CDM] vs. Warm Dark Matter [WDM]) predict distinct patterns of structure formation. CDM, for instance, allows for more small-scale clumping, while WDM suppresses such structures due to higher particle veloci

In [185]:
question_1_1 = (
"""
According to the Introduction, what motivates the use of Lyα emission from cosmic filaments as a probe of dark matter, and what specific differences between CDM and WDM do the authors expect to observe?
"""
)

In [17]:
print("=== Без RAG ===")
print(answer_without_rag(question_1_1))

=== Без RAG ===
Authors propose Lyα emission from cosmic filaments as a probe of dark matter physics because the distribution and intensity of this emission are sensitive to the small-scale structure formed by dark matter. Cold dark matter (CDM) models predict more dense, filamentary structures due to efficient gravitational collapse, enhancing gas compression and Lyα emission. Warm dark matter (WDM) models, with higher particle velocities, suppress small-scale structure formation, leading to less dense filaments and reduced Lyα emission. By observing Lyα emission in filaments, researchers can distinguish between CDM and WDM models based on the observed strength and spatial distribution of the emission, which reflects differences in dark matter particle properties and structure formation efficiency.


In [61]:
print("\n=== С RAG ===") # не совсем
print(rag_answer(question_1_1, source="1.pdf"))


=== С RAG ===
The authors motivate the use of Lyα emission from cosmic filaments as a probe of dark matter due to the sensitivity of Lyα forest features to dark matter particle properties. They argue that differences in structure formation between cold dark matter (CDM) and warm dark matter (WDM) will manifest in observable signatures in the Lyα forest, enabling constraints on dark matter models. Specifically, they expect WDM to suppress small-scale structure formation due to free-streaming of its higher-velocity particles, leading to a smoother distribution of gas in filaments compared to CDM. This suppression would alter the Lyα forest’s absorption patterns, such as reducing power on small scales or modifying the distribution of absorption line strengths, thereby distinguishing WDM from CDM. The authors emphasize that these differences arise directly from the distinct free-streaming effects and halo formation efficiencies predicted by the two models.


# 2 Seeds of supermassive black holes in general relativistic and alternative cosmologies: Implications of massive seeds

In [21]:
# 2 https://arxiv.org/pdf/2601.22991

=== Без RAG ===
The paper *"Seeds of supermassive black holes in general relativistic and alternative cosmologies: Implications of massive seeds"* explores the formation and growth of supermassive black holes (SMBHs) to their massive scales by redshift $ z \approx 10 $, which corresponds to approximately 1.3 billion years after the Big Bang. The study integrates mechanisms of black hole seed formation, growth dynamics, and cosmological models to address the challenge of how SMBHs could have formed so early in the universe. Here's a structured summary of the key mechanisms and considerations discussed:

---

### **1. Black Hole Seed Formation Mechanisms**
The paper evaluates several proposed mechanisms for the formation of the initial "seeds" of SMBHs, which are critical for their rapid growth to supermassive scales by $ z = 10 $:

#### **a. Direct Collapse of Massive Gas Clouds**
- **Mechanism**: Massive gas clouds (with masses $ \sim 10^4 - 10^6 M_\odot $) could collapse directly into

In [64]:
question_2_1 = (
"""
How do the authors calculate the minimum seed mass required to grow into a supermassive black hole at a given redshift,
and what assumptions do they make about the accretion duty cycle?
"""
)

In [67]:
print("=== Без RAG ===")
print(answer_without_rag(question_2_1))

=== Без RAG ===
Authors calculate the minimum seed mass using the Eddington luminosity and radiative efficiency (η ≈ 0.1) to estimate the accretion rate required to grow a black hole from a seed mass to its observed mass by a given redshift. They assume the black hole accretes at the Eddington limit for the available cosmic time until the redshift, solving for the initial mass:  
**M_seed = M_final - (L_Edd * η / c²) * t(z)**,  
where **t(z)** is the time since the Big Bang at redshift **z**.  

Assumptions about the accretion duty cycle include:  
1. **Duty cycle fraction**: They often assume a fraction of the cosmic time (e.g., 10–30%) during which accretion occurs, reflecting AGN activity or cosmic star formation rates.  
2. **Efficient accretion**: Some models assume continuous accretion (duty cycle ≈ 100%) for simplicity, though this may overestimate growth rates.  
3. **Radiative efficiency**: Fixed η ≈ 0.1 is commonly used, though higher efficiencies (e.g., in disk-jet scenarios

In [66]:
print("\n=== С RAG ===") # правильно
print(rag_answer(question_2_1))


=== С RAG ===
The authors calculate the minimum seed mass required to grow into a supermassive black hole (SMBH) at a given redshift using the following approach:

1. **Growth Equation**:  
   They solve the differential equation governing black hole mass growth (Equation 3):  
   $$
   \frac{dM_{\text{BH}}}{dt} = \left( \frac{f}{\epsilon} \right) \left( \frac{M_{\text{BH}}}{t_E} \right) (1 - \epsilon)
   $$  
   Here, $ f $ is the Eddington fraction (accretion rate relative to the Eddington limit), $ \epsilon $ is the accretion efficiency, and $ t_E $ is the Salpeter-Eddington timescale (~469 Myr). The solution to this equation (Equation 4) gives the seed mass $ M_s $ required to grow to a final mass $ M_{\text{BH}} $ at redshift $ z_f $:  
   $$
   f M_s = M_{\text{BH}}(t(z_f)) \exp\left( -\frac{f(1 - \epsilon)}{\epsilon} \frac{t(z_f) - t(z_i)}{t_E} \right)
   $$  
   where $ z_i $ is the initial redshift of accretion (assumed to be earlier than $ z_f $).

2. **Key Assumptions**:  


In [68]:
question_2_2 = (
    """
    According to Section 3, how do Eddington-limited and super-Eddington accretion scenarios affect
    the required seed masses for forming supermassive black holes in different cosmologies?
    """
)

In [78]:
print("=== Без RAG ===")
print(answer_without_rag(question_2_2))

print("\n=== С RAG ===")
print(rag_answer(question_2_2, source="2.pdf"))

=== Без RAG ===
In Eddington-limited accretion, the growth of supermassive black holes (SMBHs) is constrained by radiation pressure, requiring **larger seed masses** to reach SMBH masses within cosmic timescales. Conversely, super-Eddington accretion, where accretion rates exceed the Eddington limit, allows **smaller seed masses** due to faster growth. 

In different cosmologies, the required seed masses depend on factors like gas availability and structure formation efficiency. For example, in high-redshift or high-density universes, super-Eddington accretion may further reduce required seed masses, while lower gas densities or inefficient cooling could necessitate larger seeds even under super-Eddington conditions. Thus, **cosmological parameters modulate the effectiveness of accretion scenarios**, influencing the critical seed mass thresholds for SMBH formation.

=== С RAG ===
According to Section 3, **Eddington-limited and super-Eddington accretion scenarios** influence the require

# 3 On the Sidon tails of {⌊xn⌋}

In [19]:
# 3 https://arxiv.org/pdf/2601.19807

=== Без RAG ===
A **Sidon tail** of the set $ S_x = \{ \lfloor x^n \rfloor \} $ refers to a **subset of the sequence** $ \lfloor x^n \rfloor $, specifically the **tail of the sequence** (i.e., the elements for sufficiently large $ n $), which has the property of being a **Sidon sequence**. 

---

### Key Interpretation

In mathematical contexts, a **Sidon sequence** is a sequence of numbers such that all pairwise sums of its elements are distinct. That is, for any two distinct pairs $ (a_i, a_j) $ and $ (a_k, a_l) $, the sums $ a_i + a_j $ and $ a_k + a_l $ are not equal.

When applied to the set $ S_x = \{ \lfloor x^n \rfloor \} $, the **Sidon tail** refers to the **asymptotic part of the sequence** $ \lfloor x^n \rfloor $, i.e., the elements of the sequence for sufficiently large $ n $, which form a **Sidon sequence**.

---

### Why the Term "Tail" is Used

The term **tail** in this context implies that the **Sidon property is only guaranteed for large enough terms** in the sequence.

In [100]:
question_3_1 = (
"""
According to Theorem 4.1, how is the value of r chosen to be arbitrarily close to 2,
and which polynomial is used in this construction?
"""
)

In [101]:
print("=== Без RAG ===")
print(answer_without_rag(question_3_1))

=== Без RAG ===
The value of $ r $ is chosen arbitrarily close to 2 by adjusting parameters in the construction to approach this threshold, while the polynomial used is typically a **Reed-Solomon polynomial**. This construction leverages polynomial interpolation to achieve the desired properties, such as rate and distance trade-offs in coding theory. 

**Answer:**  
The polynomial used is a **Reed-Solomon polynomial**, and $ r $ is chosen arbitrarily close to 2 by optimizing the code's parameters.


In [102]:
print("\n=== С RAG ===") # очень верно
print(rag_answer(question_3_1, source="3.pdf", mode="math"))


=== С RAG ===
According to Theorem 4.1, the value of $ r $ is chosen as the unique real root $ \tilde{\alpha}_k $ of the **k-Fibonacci polynomial** $ f_k(x) = x^k - x^{k-1} - x^{k-2} - \cdots - x - 1 $. This polynomial is used in the construction because its real root $ \tilde{\alpha}_k $ approaches 2 as $ k \to \infty $ (a known property of this polynomial). For any $ \epsilon > 0 $, selecting a sufficiently large odd integer $ k $ ensures $ \tilde{\alpha}_k \in (2 - \epsilon, 2) $, making $ r = \tilde{\alpha}_k $ arbitrarily close to 2. The polynomial's structure and root behavior are critical to analyzing collisions in the sequence $ S_r $, which underpins the proof that $ S_r $ is not tail Sidon. 

**Answer:**  
The value of $ r $ is chosen as the unique real root $ \tilde{\alpha}_k $ of the **k-Fibonacci polynomial** $ f_k(x) = x^k - x^{k-1} - x^{k-2} - \cdots - x - 1 $. As $ k \to \infty $, $ \tilde{\alpha}_k \to 2 $, allowing $ r $ to be arbitrarily close to 2.


# 4 Splitting symplectic monodromy

In [168]:
# 4 https://arxiv.org/pdf/2601.20438
question_4 = (
"""
How is the group π₀ Sympᵉˣᵖ(X) defined, and which subgroup of the volume-preserving Cremona group
admits a natural map to it?
"""
)

In [166]:
print("=== Без RAG ===")
print(answer_without_rag(question_4))

=== Без RAG ===
The group $\pi_0 \text{Symp}^{\text{exact}}(X)$, where $\text{Symp}^{\text{exact}}(X)$ denotes the group of exact symplectomorphisms of a compact symplectic manifold $X$, is the group of connected components of the exact symplectomorphism group. 

In the context of a symplectic manifold where the symplectic form $\omega$ is exact, i.e., $\omega = d\theta$ for some 1-form $\theta$, the group of exact symplectomorphisms $\text{Symp}^{\text{exact}}(X)$ coincides with the group of volume-preserving diffeomorphisms of $X$. This is because the volume form associated with the symplectic form $\omega$ is preserved under symplectomorphisms, and when $\omega$ is exact, the volume form is also preserved by diffeomorphisms that preserve the exact form.

Thus, the group $\pi_0 \text{Symp}^{\text{exact}}(X)$ is equivalent to the group of connected components of the volume-preserving diffeomorphism group of $X$, denoted $\text{Diff}_\text{vol}(X)$. 

The natural map in question is fro

In [167]:
print("\n=== С RAG ===") # +-
print(rag_answer(question_4, source="4.pdf", mode="math"))


=== С RAG ===
 Refer to the content of the provided text and answer the question.
</think>

The group **π₀ Sympᵉˣᵖ(X)** is defined as the **connected components of the group of exact symplectomorphisms** of the symplectic manifold $ X $. In other words, it classifies the **path components** of the group of symplectomorphisms that preserve the exact symplectic structure of $ X $.

### Subgroup Admitting a Natural Map to π₀ Sympᵉˣᵖ(X)

The **subgroup of the volume-preserving Cremona group** that admits a natural map to $ \pi_0 \text{Symp}^\text{exp}(X) $ is the **pure braid group** $ PBr_n $, which acts on the symplectic manifold $ X $ via **symplectic automorphisms** that preserve the exact structure and the volume form.

This connection arises because the **pure braid group** can be realized as a subgroup of the **symplectic group** of $ X $, and its action on the **Lagrangian tori** (which are preserved under exact symplectomorphisms) induces a map to $ \pi_0 \text{Symp}^\text{exp}(X

# 5 Convergence of Multi-Level Markov Chain Monte Carlo Adaptive Stochastic Gradient Algorithms

In [23]:
# 5 https://arxiv.org/pdf/2601.22799

=== Без RAG ===
The **multilevel Markov chain Monte Carlo (MCMC) framework** introduced in the paper *"Convergence of Multi-Level Markov Chain Monte Carlo Adaptive Stochastic Gradient Algorithms"* combines **multilevel Monte Carlo (MLMC) techniques** with **adaptive stochastic gradient methods** to improve convergence rates in optimization problems involving expectations over complex distributions. Here's a structured breakdown of its key components and how it enhances convergence:

---

### **1. Core Components of the Framework**
- **Multilevel MCMC**:  
  The framework leverages a hierarchy of MCMC samplers at different "levels," where each level corresponds to a discretization of the problem (e.g., varying resolution or computational cost). Coarse-level samplers are computationally cheaper but less accurate, while fine-level samplers are more accurate but more expensive.  
  - **Goal**: Combine results from multiple levels to approximate expectations (e.g., gradients) with **reduced

In [158]:
question_5_1 = (
"""
Why is the IWAE gradient estimator biased as a self-normalized importance sampling (SNIS) estimator for finite k?
What is the order of this bias, and how do BR-SNIS and MLMC-IWAE reduce it according to Section 4?
"""
)

In [159]:
print("=== Без RAG ===")
print(answer_without_rag(question_5_1))

=== Без RAG ===
The IWAE gradient estimator is biased as a self-normalized importance sampling (SNIS) estimator for finite $ k $ because the normalization term (sum of importance weights) is estimated using the same samples, introducing approximation errors. The bias is of order $ \mathcal{O}(1/k) $.

**BR-SNIS** reduces this bias by explicitly correcting the normalization approximation through bias-reduction techniques, such as adjusting weights or using higher-order terms. **MLMC-IWAE** reduces the bias by combining estimators at different levels of sampling (coarse and fine), leveraging the multilevel Monte Carlo framework to cancel bias terms and improve convergence. Both methods mitigate the $ \mathcal{O}(1/k) $ bias by addressing the finite-sample normalization error.


In [160]:
print("\n=== С RAG ===")
print(rag_answer(question_5_1, source="5.pdf", mode="general"))


=== С RAG ===
The IWAE gradient estimator is biased as a self-normalized importance sampling (SNIS) estimator for finite $ k $ due to the inherent limitations of estimating the normalization term with a finite number of samples. In SNIS, the importance weights are normalized by their sum, which is itself an estimator. This normalization introduces **bias** because the expectation of the normalized weights does not exactly match the true value. Specifically, the gradient estimator involves terms like $ \mathbb{E}[\log(\sum w_i)] $, where $ \sum w_i $ is an estimated normalization constant. Since this sum is approximated with finite samples, the gradient of the estimator becomes biased, as the true expectation of the log normalization term is not captured exactly.

### Order of Bias
The bias of the IWAE gradient estimator is typically **of order $ O(1/k) $**. This arises because the variance of the normalization term decreases as $ k $ increases, but the bias remains proportional to the

# 6 HetCCL: Accelerating LLM Training with Heterogeneous GPUs

In [24]:
# 6 https://arxiv.org/pdf/2601.22585
question_6 = "What is the HetCCL framework introduced in the paper “HetCCL: Accelerating LLM Training with Heterogeneous GPUs,” and how does it improve cross‑vendor GPU training performance for large language models?"


print("=== Без RAG ===")
print(answer_without_rag(question_6))


print("\n=== С RAG ===")
print(rag_answer(question_6))

=== Без RAG ===
The **HetCCL framework** (introduced in the paper *"HetCCL: Accelerating LLM Training with Heterogeneous GPUs"*) is designed to optimize the training of large language models (LLMs) on **mixed-vendor GPU clusters**, where GPUs from different manufacturers (e.g., NVIDIA, AMD, or others) are used together. The key challenge in such environments is **communication inefficiency** due to differences in hardware capabilities, interconnects, and proprietary communication protocols. HetCCL addresses these challenges by enabling **seamless, high-performance training across heterogeneous GPUs**, thereby improving scalability and cost-effectiveness.

---

### **Key Features and Contributions of HetCCL**

1. **Unified Communication Abstraction**  
   HetCCL provides a **single API** that abstracts the differences between GPU vendors. It integrates multiple communication backends (e.g., NVIDIA's **NCCL**, AMD's **ROCm**, or other proprietary libraries) into a unified interface. This

In [139]:
question_6_1 = (
    """
    According to Section 2, what are the main limitations of existing collective communication libraries (CCLs) for GPUs,
    and why do they fail to support cross-vendor GPU communication?
    """
)

# What are the differences between intra-node and inter-node GPU communication
# mechanisms, and how does RDMA improve inter-node GPU communication according?

In [140]:
print("=== Без RAG ===")
print(answer_without_rag(question_6_1))

print("\n=== С RAG ===")
print(rag_answer(question_6_1, source="6.pdf"))

=== Без RAG ===
Existing collective communication libraries (CCLs) for GPUs face limitations due to **vendor-specific APIs and hardware optimizations**, which prevent cross-vendor compatibility. They are tailored to proprietary interconnects (e.g., NVIDIA's NVLink, AMD's Infinity Fabric) and lack standardized abstractions. This results in **incompatibility** with other vendors' hardware, as their communication protocols, memory architectures, and performance optimizations are not universally supported.

=== С RAG ===
The main limitations of existing collective communication libraries (CCLs) for GPUs, as outlined in Section 2, are:

1. **Vendor-Specific Optimization and Integration**:  
   Existing CCLs (e.g., NCCL for NVIDIA, RCCL for AMD) are tightly coupled with specific GPU microarchitectures and include extensive vendor-specific optimizations. This makes them highly efficient in homogeneous environments but incompatible with cross-vendor setups. For example, NCCL and RCCL are deepl

In [169]:
print("\n=== С RAG ===")
print(rag_answer(question_6_1, source="6.pdf", mode="math"))


=== С RAG ===
The main limitations of existing collective communication libraries (CCLs) for GPUs, as outlined in the text, are:  

1. **Vendor-Specific Design**: Existing CCLs like **NCCL** (for NVIDIA) and **RCCL** (for AMD) are tightly coupled to specific GPU hardware and software stacks. They rely on proprietary drivers, optimizations, and runtime APIs tailored to individual vendors, which are incompatible with other vendors' hardware. This creates a **fragmented ecosystem** where cross-vendor communication is not supported.  

2. **Performance Optimization for Homogeneous Environments**: These libraries are optimized for **homogeneous GPU clusters** (e.g., all NVIDIA or all AMD GPUs). They leverage deep integration with GPU microarchitectures and vendor-specific features (e.g., CUDA or HIP). However, this optimization **does not extend to heterogeneous setups**, where performance gaps arise due to differences in hardware capabilities and communication protocols.  

3. **Lack of C

# 7 Stable Personas: Dual-Assessment of Temporal Stability in LLM-Based Human Simulation

In [29]:
# 7 https://arxiv.org/pdf/2601.22812
question_7 = "What dual‑assessment framework do the authors introduce in “Stable Personas: Dual‑Assessment of Temporal Stability in LLM‑Based Human Simulation,” and what did they find about the stability of LLM‑generated personas over extended conversations?"


print("=== Без RAG ===")
print(answer_without_rag(question_7))


print("\n=== С RAG ===")
print(rag_answer(question_7))

=== Без RAG ===
In the paper **"Stable Personas: Dual-Assessment of Temporal Stability in LLM-Based Human Simulation"**, the authors introduce a **dual-assessment framework** to evaluate the **temporal stability** of personas generated by large language models (LLMs) during extended conversations. This framework is designed to address the challenge of maintaining consistent character traits, behaviors, and identities over time in simulated human interactions.

### **Dual-Assessment Framework**
The authors propose two complementary methods to assess temporal stability:

1. **Internal Consistency Assessment**:  
   This evaluates how well the persona's responses align with its **initial traits and goals** over time. It focuses on **automated metrics** (e.g., consistency in responses, adherence to predefined personality attributes) and **model-based analysis** (e.g., tracking deviations from the persona's core characteristics across interactions).

2. **External Validation Assessment**:  

In [170]:
question_7_1 = (
    """
According to Experiment I, which persona intensity condition (low, moderate, high) 
shows the highest and lowest between-conversation stability based on the reported standard deviations,
and which factor accounts for the largest proportion of variance in self-reported and observer-rated scores?"
    """
)

In [171]:
print("=== Без RAG ===")
print(answer_without_rag(question_7_1))

=== Без RAG ===
In Experiment I:  
- **Highest between-conversation stability** (lowest standard deviation) was observed in the **low intensity** condition.  
- **Lowest stability** (highest standard deviation) was in the **high intensity** condition.  
- The **persona intensity condition** accounted for the largest proportion of variance in both **self-reported** and **observer-rated** scores.


In [174]:
print("\n=== С RAG ===")
print(rag_answer(question_7_1, source="7.pdf", mode="general"))


=== С RAG ===
Based on Experiment I's reported standard deviations and variance analysis:

1. **Between-Conversation Stability**:  
   - **Highest stability**: **High intensity** (standard deviation = 0.35).  
   - **Lowest stability**: **Low intensity** (standard deviation = 0.51).  
   - **Moderate intensity** (standard deviation = 0.42) falls between the two.  

2. **Factor accounting for the largest variance**:  
   The **persona intensity condition** (high, moderate, low) accounts for the largest proportion of variance in both self-reported and observer-rated scores. This suggests that the intensity level is the primary driver of differences in the scores across conditions.  

**Answer**:  
- **Highest stability**: High intensity (SD = 0.35).  
- **Lowest stability**: Low intensity (SD = 0.51).  
- **Largest variance contributor**: The **persona intensity condition** (high, moderate, low).


In [141]:
question_7_2 = (
    """
    How do low-, moderate-, and high-intensity personas differ in terms of between-conversation stability,
    according to the reported standard deviations in Experiment I?
    """
)

In [142]:
print("=== Без RAG ===")
print(answer_without_rag(question_7_2))

print("\n=== С RAG ===")
print(rag_answer(question_7_2, source="7.pdf"))

=== Без RAG ===
According to Experiment I, low-intensity personas exhibited the lowest between-conversation stability (smallest standard deviations), moderate-intensity personas showed intermediate stability, and high-intensity personas demonstrated the greatest instability (largest standard deviations). This reflects varying degrees of behavioral consistency across conversations.

=== С RAG ===
The standard deviations for between-conversation stability in Experiment I reveal differences in response consistency across low-, moderate-, and high-intensity personas. Based on the data:

- **Low-intensity personas** (e.g., "Low" in the table) exhibit the **lowest standard deviations** (e.g., 0.5 for Observer, 0.7 for Self), indicating **high consistency** in responses across conversations.  
- **Moderate-intensity personas** (e.g., "Moderate") show **intermediate standard deviations** (e.g., 0.8 for Observer, 1.0 for Self), reflecting **moderate variability** in responses.  
- **High-intens

# 8 Toward Non-Expert Customized Congestion Control

In [30]:
# 8 https://arxiv.org/pdf/2601.22461
question_8 = "What framework does the paper “Toward Non-Expert Customized Congestion Control” propose for enabling non-expert users to design customized congestion control algorithms, and how does it leverage Large Language Models and the Berkeley Packet Filter interface to accomplish this?"


print("=== Без RAG ===")
print(answer_without_rag(question_8))


print("\n=== С RAG ===")
print(rag_answer(question_8))

=== Без RAG ===
The paper **"Toward Non-Expert Customized Congestion Control"** proposes a framework that enables non-expert users to design and customize congestion control algorithms by combining **Large Language Models (LLMs)** with the **Berkeley Packet Filter (BPF)** interface. Here's a breakdown of the framework and its components:

---

### **1. Framework Overview:**
The framework aims to **democratize congestion control customization** by abstracting the technical complexity of network protocols. It allows users (even non-experts) to define their desired congestion control behavior using **natural language** (e.g., "prioritize low latency" or "avoid high packet loss"). The system then translates these high-level preferences into **customized congestion control algorithms** and deploys them in the network stack.

---

### **2. Key Components and Mechanisms:**

#### **A. Large Language Models (LLMs) for Natural Language to Algorithm Translation**
- **Role**: The LLM acts as a **t

In [143]:
question_8_1 = (
    """
    What limitations of existing automated congestion control and networking code generation approaches
    motivate the authors to target non-expert users?
    """
)

In [144]:
print("=== Без RAG ===")
print(answer_without_rag(question_8_1))

print("\n=== С RAG ===")
print(rag_answer(question_8_1, source="8.pdf"))

=== Без RAG ===
Existing approaches often require deep expertise in networking protocols, limiting accessibility to non-experts. They may also lack flexibility, scalability, or user-friendly interfaces, making manual intervention or customization necessary. These limitations hinder broad adoption and motivate the development of tools tailored for non-expert users, emphasizing simplicity, adaptability, and ease of use.

=== С RAG ===
The authors identify several limitations in existing automated congestion control (CC) and networking code generation approaches that motivate their focus on non-expert users:

1. **Inadequate Code Context and References**:  
   Existing methods often lack sufficient domain-specific code references (e.g., Linux kernel TCP implementations). Prompts are constrained by size and complexity, leading to incomplete information. This can cause LLMs to misunderstand critical components (e.g., misadjusting pacing rates instead of congestion window sizes), resulting i

# 9 Minimal-Action Discrete Schrödinger Bridge Matching for Peptide Sequence Design

In [27]:
# 9 https://arxiv.org/pdf/2601.22408
question_9 = "What is the Minimal‑Action Discrete Schrödinger Bridge Matching (MadSBM) framework introduced in the paper “Minimal‑Action Discrete Schrödinger Bridge Matching for Peptide Sequence Design,” and how does it improve generative modeling of peptide sequences compared to discrete diffusion and flow‑based methods?"


print("=== Без RAG ===")
print(answer_without_rag(question_9))


print("\n=== С RAG ===")
print(rag_answer(question_9))

=== Без RAG ===
The **Minimal-Action Discrete Schrödinger Bridge Matching (MadSBM)** framework, introduced in the paper *“Minimal-Action Discrete Schrödinger Bridge Matching for Peptide Sequence Design,”* is a novel generative modeling approach for designing peptide sequences. It leverages concepts from **optimal transport** and **Schrödinger bridges** to efficiently generate sequences that match target distributions while minimizing a cost function (action) that encodes biological or structural constraints. Here's a breakdown of its key components and advantages:

---

### **Core Principles of MadSBM**
1. **Schrödinger Bridges**:  
   - These are a class of probability distributions that describe the most probable paths between two distributions under a given stochastic dynamics. In the discrete case, they model transitions between states (e.g., amino acids) with minimal "action," which is analogous to minimizing energy or cost in physics.
   - The framework uses **discrete Schrödinge

In [145]:
question_9_1 = (
    """
    How does MadSBM compare to the discrete diffusion baseline (EvoFlow)
    in terms of unconditional peptide sequence generation quality across different sampling budgets?
    """
)

In [146]:
print("=== Без RAG ===")
print(answer_without_rag(question_9_1))

print("\n=== С RAG ===")
print(rag_answer(question_9_1, source="9.pdf"))

=== Без RAG ===
MadSBM generally outperforms EvoFlow in unconditional peptide sequence generation quality, particularly with sufficient sampling budgets. MadSBM's score-based approach enables higher-quality samples at comparable computational costs, while EvoFlow's discrete diffusion process may require more steps to achieve similar quality. However, EvoFlow can maintain reasonable quality with fewer steps, depending on the specific sampling budget constraints.

=== С RAG ===
MadSBM demonstrates superior performance compared to the discrete diffusion baseline (EvoFlow) in unconditional peptide sequence generation, particularly across varying sampling budgets. Key insights from the text include:

1. **Actional Control and Efficiency**:  
   MadSBM's instantaneous actionals (AL(u)) remain consistently **below the worst-case bounds** reported in Table 3 across all sampling budgets (e.g., 1/32, 1/64, 1/128). This indicates that MadSBM minimizes the control cost required to tilt the referen

In [175]:
question_9_2 = (
    """
    What are the observed pseudo-perplexity (PPL) and pLDDT scores
    for MadSBM and the discrete diffusion baseline (EvoFlow)
    in unconditional peptide sequence generation across different sampling budgets?
    """
)

In [176]:
print("=== Без RAG ===")
print(answer_without_rag(question_9_2))

=== Без RAG ===
The observed pseudo-perplexity (PPL) and pLDDT scores for MadSBM and EvoFlow in unconditional peptide sequence generation across different sampling budgets are not explicitly detailed in publicly available datasets or benchmarks. These metrics depend on specific experimental setups and evaluation protocols, which vary across studies. For precise values, refer to the original research papers or experimental reports comparing these models.


In [177]:
print("\n=== С RAG ===")
print(rag_answer(question_9_2, source="9.pdf"))


=== С RAG ===
The observed pseudo-perplexity (PPL) and predicted local distance difference test (pLDDT) scores for **MadSBM** and the **discrete diffusion baseline (EvoFlow)** in unconditional peptide sequence generation are analyzed across different sampling budgets (N ∈ {32, 64, 128}). Here's the summary:

1. **Pseudo-Perplexity (PPL):**  
   - Both **MadSBM** and the **discrete diffusion (DD) baseline** generate peptide sequences with **lower average PPLs** than the held-out test set, which serves as a proxy for the natural peptide sequence distribution.  
   - **MadSBM** achieves **lower or competitive PPLs** compared to the DD baseline across all sampling budgets, indicating stronger alignment with the biological peptide sequence space.  
   - Specific numerical values for PPL are referenced in **Table 6** (not included here), but the text highlights that MadSBM outperforms or matches the DD baseline in terms of PPL.

2. **pLDDT Scores (Structural Stability):**  
   - The **pLDDT

In [178]:
question_9_3 = ("""How is generative sampling performed after training, and how are transition rates, jump probabilities, and token updates defined in the CTMC-based sampling procedure?""")

In [179]:
print("=== Без RAG ===")
print(answer_without_rag(question_9_3))

=== Без RAG ===
Generative sampling in CTMC-based models involves simulating transitions between states (tokens) using predefined transition rates. After training, the model's parameters define these rates, which govern the likelihood of moving from one state to another. 

- **Transition rates**: Determined during training, these rates (e.g., $ q_{ij} $) represent the instantaneous probability of transitioning from state $ i $ to state $ j $.  
- **Jump probabilities**: Computed as $ \frac{q_{ij}}{\sum_{k \neq i} q_{ik}} $, normalizing the transition rates to ensure valid probabilities.  
- **Token updates**: At each step, the next token is selected based on the jump probabilities, and the process repeats until termination (e.g., end-of-sequence token or maximum length).  

The CTMC framework ensures that sampling respects the continuous-time dynamics of the model.


In [181]:
print("\n=== С RAG ===")
print(rag_answer(question_9_3, source="9.pdf", mode = "general"))


=== С RAG ===
Generative sampling in MadSBM is performed using a **controlled continuous-time Markov chain (CTMC)** framework, where the process evolves backward in time from a fully masked prior to the data distribution. Here's a breakdown of the key components:

---

### **1. Sampling Initialization**
- **Starting Point**: Sampling begins from the **fully masked prior** $ \mu_0 = \{[MASK]\}_L $, where all tokens are initially masked (e.g., placeholders for amino acids).
- **Direction**: The CTMC is simulated **backward in time** from $ t = 1 $ (fully masked state) to $ t = 0 $ (data distribution), following the learned generator $ R_\theta $.

---

### **2. Transition Rates and Jump Probabilities**
- **Generator $ R_\theta $**: The CTMC is driven by the **learned generator** $ R_\theta $, which encodes the time-dependent control field $ u_\theta $. This generator defines the **transition rates** between states (token sequences) in the amino-acid edit graph.
  - **Minimal-Action Path

# 10 Endogenous Inequality Aversion: Decision criteria for triage and other ethical tradeoffs

In [28]:
# 10 https://arxiv.org/pdf/2601.22250
question_10 = "What axiomatic framework do the authors propose in “Endogenous Inequality Aversion: Decision criteria for triage and other ethical tradeoffs,” and how does it explain shifting between utilitarian and Rawlsian principles in allocating scarce resources?"


print("=== Без RAG ===")
print(answer_without_rag(question_10))


print("\n=== С RAG ===")
print(rag_answer(question_10))

=== Без RAG ===
The authors of *“Endogenous Inequality Aversion: Decision criteria for triage and other ethical tradeoffs”* propose an **axiomatic framework** that integrates **endogenous inequality aversion** into decision-making under scarcity, allowing for the **flexible application of utilitarian and Rawlsian principles** depending on the context. Here's a structured breakdown of their approach:

---

### **1. Core Axiomatic Framework: Endogenous Inequality Aversion**
The framework is built on the idea that **inequality aversion is not fixed but context-dependent**, shaped by the **specific circumstances of resource allocation** (e.g., urgency, scale, or distributional impacts). Key elements of the framework include:

- **Parametric Flexibility**: The decision rule incorporates a **parameter** (e.g., a weight on inequality) that adjusts based on the **context of the allocation problem**. This parameter determines how much priority is given to reducing inequality versus maximizing t

In [147]:
question_10_1 = (
    """
    How do the mixing and homotheticity axioms in the model lead to the representation of 
    social welfare through a “fan” function with weights endogenously dependent on the level of welfare?
    """
)

In [148]:
print("=== Без RAG ===")
print(answer_without_rag(question_10_1))

print("\n=== С RAG ===")
print(rag_answer(question_10_1, source="10.pdf"))

=== Без RAG ===
The mixing and homotheticity axioms lead to a "fan" function representation of social welfare by ensuring that weights assigned to individual utilities are endogenous to welfare levels. 

- **Mixing** requires social welfare to be a convex combination of individual utilities, implying weights depend on the relative importance of different utility profiles.  
- **Homotheticity** enforces that the weights are homogeneous of degree one, meaning they scale proportionally with the overall welfare level.  

Together, these axioms imply that the weights in the social welfare function are functions of the total welfare, creating a "fan" structure where the weights adapt endogenously to changes in the welfare level. This ensures the representation is both flexible and consistent with the axioms.

=== С RAG ===
The mixing and homotheticity axioms in the model lead to the representation of social welfare through a "fan" function with endogenously dependent weights by ensuring cons

In [183]:
print("\n=== С RAG ===") # 7 лучший
print(rag_answer(question_10_1, mode = "stat"))


=== С RAG ===
The mixing and homotheticity axioms in the model lead to the representation of social welfare through a "fan" function with endogenously dependent weights by structuring the trade-off between utilitarianism and inequality aversion in response to varying welfare levels. Here's how:

---

### **1. Mixing Invariance and Convexity**
- **Mixing invariance** ensures that the social welfare function is **convex** and **invariant to the mixing of policies**. This means the planner’s preference over a combination of two policies depends only on the welfare levels of the individual policies, not their specific composition. 
- This convexity requirement naturally aligns with a **fan function** (a set of weight distributions over outcomes) because it allows the planner to interpolate between different weightings (e.g., utilitarian vs. Rawlsian) depending on the context. The fan function captures this by defining a family of weights that are consistent with the convexity of the socia